In [ ]:

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer


In [5]:
load_dotenv()

True

In [ ]:
llm= ChatGroq(model="openai/gpt-oss-120b")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
loader=PyPDFLoader("sn.pdf")
docs=loader.load()


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-02-03T11:12:34+05:30', 'author': 'Un-named', 'moddate': '2026-02-03T11:12:34+05:30', 'source': 'sn.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='PROJECT SYNOPSIS \n-------------------------------------------------------------------------------------------- \nShashank Nautiyal \nRoll No.: 241347080028 \nEnrollment Number: G211341849 \n-------------------------------------------------------------------------------------------- \nSubmitted To: Prof. Y. P. Raiwani \nDepartment of Computer Science & Engineering \nHemvati Nandan Bahuguna Garhwal University (HNBGU) \n-------------------------------------------------------------------------------------------- \nTITLE \nAI-Powered Multi-Agent Application Builder Using LangGraph \n \n1. Introduction \nIn modern software development, creating applications from scratch requires significant \ntime, expertise, and i

In [9]:
text_spliter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_spliter.split_documents(docs)

In [ ]:
vector_store = FAISS.from_documents(chunks, model)

In [ ]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
@tool
def rag_tool(query):

    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):

    messages = state['messages']

    response = llm_with_tools.invoke(messages)

    return {'messages': [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()

In [ ]:
chatbot

In [ ]:
result = chatbot.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "using the pdf explain the architecture of project"
                )
            )
        ]
    }
)

In [ ]:
print(result['messages'][-1].content)